# Ciclo DENIT GWF-GWT-Interfaz

Este notebook ejecuta el ciclo Denit usando:

- `u_init.dat`
- `conc_i.m3d`
- `u_tilde.dat`
- `u_new.out`
- `GWF-V5.ipynb`
- `GWT-V5.ipynb`
- `interfaz_remix.exe`

La primera etapa usa las líneas `24+m` y `44+m` de `u_init.dat`.

Después de ejecutar la interfaz, las etapas siguientes usan las líneas `3+m` de `u_new.out`.


## Secuencia implementada

1. Lee `u_init.dat`.
2. Lee la línea 24 y la línea 44.
3. Guarda ambas líneas como columna.
4. Copia la primera columna a `conc_i.m3d`.
5. Define globalmente `conc_inicial` y `contenido_44`.
6. Ejecuta `GWF-V5.ipynb` y `GWT-V5.ipynb`.
7. Obtiene `cobj = o_gwt.output.concentration()`.
8. Sustituye el renglón correspondiente de `u_tilde.dat`.
9. Repite lo anterior para `n_comp`.
10. Ejecuta `interfaz_remix.exe < datos_interfaz_denit_2reacts`.
11. Lee `u_new.out` desde la línea 3.
12. Repite el ciclo para `nv` pasos.


In [1]:
def detener():
    print("\n⏹ Ejecución detenida por el usuario")
    raise KeyboardInterrupt

In [2]:
###############################################################
# BLOQUE 1. IMPORTACIÓN DE LIBRERÍAS
###############################################################

import os
import time
import numpy as np
import pandas as pd
import ipywidgets as widgets
from IPython.display import display
from IPython import get_ipython

###############################################################
# BLOQUE 2. CONTROL DE REPETICIONES
###############################################################

n_comp = 7   # número de veces a ejecutar el proceso = num_comp del sistema
nv     = 10   # número de pasos de tiempo
nplot  = 1   # frecuencia de visualización

ncolt  = 10  # ncolt = número total de columnas del dominio
dt = 1e-1    # duración del paso de tiempo
dely = 0.10   # delr ancho de la celda dirección y
delx = 0.10   # delc ancho de la celda dirección x
delz = 0.10   # top alto de la celda dirección z

imprimir = True

###############################################################
# BLOQUE 3. ARCHIVOS PRINCIPALES
###############################################################

archivo_u_init  = "u_init.dat"
archivo_conci   = "conc_i.m3d"
archivo_u_tilde = "u_tilde.dat"
archivo_u_new   = "u_new.out"
archivo_u_new1  = "u_new1.out"

interfaz = "interfaz_remix.exe"
archivo_entrada_interfaz = "datos_interfaz_denit_2reacts"

comando = f"{interfaz} < {archivo_entrada_interfaz}"

###############################################################
# BLOQUE 4. VARIABLES GLOBALES
###############################################################

conc_inicial = None
contenido_44 = None
ntpt = 0

###############################################################
# BLOQUE 5. FUNCIONES AUXILIARES DE ARCHIVOS
###############################################################

def leer_lineas_archivo(nombre_archivo):
    """
    Lee todas las líneas de un archivo de texto.
    """
    with open(nombre_archivo, "r") as f:
        return f.readlines()


def guardar_linea_como_columna(linea, archivo_salida="conc_i.m3d"):
    """
    Toma una línea con varios valores y la guarda como columna.
    Un valor por renglón.
    """
    valores = linea.split()

    with open(archivo_salida, "w") as f:
        for v in valores:
            f.write(v + "\n")

    print("---------------------------------------------------")
    print(f"Archivo generado: {archivo_salida}")
    print(f"Número de valores guardados: {len(valores)}")
    print("Valores:")
    for i, v in enumerate(valores, start=1):
        print(f"{i:4d} -> {v}")
    print("---------------------------------------------------")

    return valores


def guardar_valores_como_renglon(valores, archivo_salida):
    """
    Guarda una lista de valores en un solo renglón.
    """
    with open(archivo_salida, "w") as f:
        f.write(" ".join(map(str, valores)) + "\n")


def leer_columna_como_tokens(archivo_entrada):
    """
    Lee un archivo que puede estar en columna o renglón y devuelve todos los tokens.
    """
    tokens = []

    with open(archivo_entrada, "r") as f:
        for line in f:
            s = line.strip()
            if s:
                tokens.extend(s.split())

    return tokens


def copiar_columna_a_conci(archivo_columna, archivo_conci="conc_i.m3d"):
    """
    Copia una columna ya guardada hacia conc_i.m3d.
    """
    tokens = leer_columna_como_tokens(archivo_columna)

    with open(archivo_conci, "w") as f:
        for v in tokens:
            f.write(v + "\n")

    print(f"✅ {archivo_conci} actualizado desde {archivo_columna}")
    print(f"   Total de valores: {len(tokens)}")

    return tokens


def linea_a_arreglo_3d(linea, ncolt=10):
    """
    Convierte una línea de texto a un arreglo con forma (1, 1, ncolt).
    """
    datos = np.array(list(map(float, linea.split())))

    if len(datos) != ncolt:
        raise ValueError(
            f"El número de datos es {len(datos)}, pero ncolt = {ncolt}."
        )

    return datos.reshape((1, 1, ncolt))


###############################################################
# BLOQUE 6. PREPARAR COMPONENTE DESDE u_init.dat
###############################################################

def preparar_componente_desde_u_init(m):
    """
    Para cada componente m:

    m = 0 -> lee línea 24 y línea 44
    m = 1 -> lee línea 25 y línea 45
    m = 2 -> lee línea 26 y línea 46
    etc.

    Guarda:
    - la línea 24+m como columna
    - la línea 44+m como columna
    - copia la primera columna a conc_i.m3d
    - define global conc_inicial
    - define global contenido_44
    """

    global conc_inicial
    global contenido_44

    nri = 24 + m
    nrj = 44 + m

    print("\n" + "="*70)
    print(f"COMPONENTE {m+1} de {n_comp}")
    #print(f"Línea inicial  = {nri}")
    #print(f"Línea fuente   = {nrj}")
    print("="*70)

    lineas = leer_lineas_archivo(archivo_u_init)

    if len(lineas) < nrj:
        raise ValueError(
            f"{archivo_u_init} no tiene suficientes líneas. "
            f"Se requiere al menos la línea {nrj}."
        )

    linea_inicial = lineas[nri - 1]
    linea_fuente  = lineas[nrj - 1]

    """
    archivo_col_inicial = f"columna_linea_{nri}.dat"
    archivo_col_fuente  = f"columna_linea_{nrj}.dat"

    print(f"\nGuardando línea {nri} como columna:")
    valores_inicial = guardar_linea_como_columna(
        linea_inicial,
        archivo_salida=archivo_col_inicial
    )

    print(f"\nGuardando línea {nrj} como columna:")
    valores_fuente = guardar_linea_como_columna(
        linea_fuente,
        archivo_salida=archivo_col_fuente
    )

    ###########################################################
    # Tomar la primera columna y copiarla a conc_i.m3d
    ###########################################################

    copiar_columna_a_conci(
        archivo_columna=archivo_col_inicial,
        archivo_conci=archivo_conci
    )
    """
    # Se sustiuyó lo comentado por estos 3 comandos
    #print(f"\nProcesando línea {nri}")
    valores_inicial = [float(x) for x in linea_inicial.split()]
    
    #print(f"\nProcesando línea {nrj}")
    valores_fuente = [float(x) for x in linea_fuente.split()]
    
    with open(archivo_conci, "w") as f:
        for v in valores_inicial:
            f.write(f"{v}\n")

    ###########################################################
    # Definir variables globales para GWT-V5.ipynb
    ###########################################################

    conc_inicial = linea_a_arreglo_3d(linea_inicial, ncolt=ncolt)

    # Se toma el primer valor de la línea 44+m como contenido_44.
    # Esto mantiene la convención usada en el notebook de referencia.
    contenido_44 = float(valores_fuente[0])

    globals()["conc_inicial"] = conc_inicial
    globals()["contenido_44"] = contenido_44

    if imprimir:
        print("\nconc_inicial =")
        print(" ".join(f"{v:.6e}" for v in conc_inicial.flatten()))

        print("\ncontenido_44 =")
        print(f"{contenido_44:.6e}")

    return valores_inicial, valores_fuente


###############################################################
# BLOQUE 7. OBTENER CONCENTRACIÓN FINAL DE GWT-V5
###############################################################

def obtener_concentracion_final_gwt():
    """
    De la salida de GWT-V5 obtiene el objeto de concentraciones:

        cobj = o_gwt.output.concentration()

    Luego toma el último tiempo disponible y devuelve la concentración
    como vector 1D.
    """

    try:
        cobj = o_gwt.output.concentration()

        times = np.array(cobj.get_times())
        print("\nTiempos disponibles en GWT:")
        print(times)

        if times.size == 0:
            print("⚠️ No hay tiempos en la salida GWT.")
            return None

        t_final = times[-1]
        conc = cobj.get_data(totim=t_final)

        conc_flat = np.asarray(conc).ravel()

        print(f"✅ Concentración final obtenida para t = {t_final}")
        print(f"   Tamaño del vector: {conc_flat.size}")

        #if imprimir:
        #    print("   Valores:")
        #    print(conc_flat)

        if imprimir:
            print("   Valores:")
            print(np.array2string(
                conc_flat,
                formatter={'float_kind': lambda x: f"{x:.6e}"}
            ))

        return conc_flat

    except Exception as e:
        print(f"❌ Error al obtener concentración desde o_gwt: {e}")
        return None


###############################################################
# BLOQUE 8. ACTUALIZAR u_tilde.dat
###############################################################

def sustituir_renglon_u_tilde_desde_valores(
    valores,
    archivo_u_tilde="u_tilde.dat",
    m=0
):
    """
    Sustituye el renglón m+1 de u_tilde.dat usando un vector de valores.
    """

    if valores is None:
        print("⚠️ No hay valores para actualizar u_tilde.dat.")
        return False

    valores = np.asarray(valores).ravel()

    if valores.size == 0:
        print("⚠️ El vector de valores está vacío.")
        return False

    linea_nueva = " ".join(map(str, valores)) + "\n"

    try:
        with open(archivo_u_tilde, "r") as f:
            lineas = f.readlines()
    except FileNotFoundError:
        lineas = []

    while len(lineas) <= m:
        lineas.append("\n")

    lineas[m] = linea_nueva

    with open(archivo_u_tilde, "w") as f:
        f.writelines(lineas)

    print(f"✅ Renglón {m+1} de {archivo_u_tilde} sustituido.")
    print(f"   Número de valores escritos: {valores.size}")

    return True


def sustituir_renglon_u_tilde_desde_conci(
    archivo_u_tilde="u_tilde.dat",
    archivo_conci="conc_i.m3d",
    m=0
):
    """
    Lee conc_i.m3d, lo colapsa a un renglón y sustituye el renglón m+1
    de u_tilde.dat.
    """

    tokens = leer_columna_como_tokens(archivo_conci)

    return sustituir_renglon_u_tilde_desde_valores(
        valores=tokens,
        archivo_u_tilde=archivo_u_tilde,
        m=m
    )


###############################################################
# BLOQUE 9. EJECUTAR GWF-V5 Y GWT-V5
###############################################################

def ejecutar_gwf_gwt(ip, m, i):
    """
    Ejecuta GWF-V5.ipynb y GWT-V5.ipynb.

    Después obtiene la concentración de salida de GWT.
    """

    global ntpt

    ntpt = i
    globals()["ntpt"] = ntpt
    globals()["m"] = m
    globals()["n_comp"] = n_comp
    globals()["nv"] = nv
    globals()["nplot"] = nplot
    globals()["conc_inicial"] = conc_inicial
    globals()["contenido_44"] = contenido_44

    print("\n▶️ Ejecutando GWF-V5.ipynb ...")
##    ip.run_line_magic("run", "GWF-V5.ipynb")

    print("\n▶️ Ejecutando GWT-V5.ipynb ...")
    ip.run_line_magic("run", "GWT-V5.ipynb")

    conc_flat = obtener_concentracion_final_gwt()

    return conc_flat


###############################################################
# BLOQUE 10. EJECUTAR INTERFAZ
###############################################################

def ejecutar_interfaz():
    """
    Ejecuta:

        interfaz_remix.exe < datos_interfaz_denit_2reacts

    usando el comando:

        comando = f"{interfaz} < {archivo_entrada_interfaz}"
    """

    print("\n" + "#"*70)
    print("# EJECUTANDO INTERFAZ")
    print("#"*70)
    print(f"comando = {comando}\n")

    ret = os.system(comando)

    print(f"\nCódigo de retorno de interfaz: {ret}")

    return ret


###############################################################
# BLOQUE 11. PREPARAR COMPONENTE DESDE u_new.out
###############################################################

def preparar_componente_desde_u_new(m):
    """
    Para cada componente m:

    m = 0 -> lee línea 3 de u_new.out
    m = 1 -> lee línea 4 de u_new.out
    m = 2 -> lee línea 5 de u_new.out
    etc.

    Convierte esa línea en columna y la guarda en conc_i.m3d.
    """

    nr = 3 + m

    print("\n" + "="*70)
    print(f"LECTURA DESDE u_new.out")
    print(f"Componente {m+1} de {n_comp}")
    print(f"Línea leída = {nr}")
    print("="*70)

    lineas = leer_lineas_archivo(archivo_u_new)

    if len(lineas) < nr:
        raise ValueError(
            f"{archivo_u_new} no tiene suficientes líneas. "
            f"Se requiere al menos la línea {nr}."
        )

    linea = lineas[nr - 1]

    with open(archivo_u_new1, "w") as f:
        f.write(linea.rstrip("\n") + "\n")

    print(f"✅ Línea {nr} copiada a {archivo_u_new1}")

    valores = guardar_linea_como_columna(
        linea,
        archivo_salida=archivo_conci
    )

    print(f"✅ Línea {nr} convertida en columna y guardada en {archivo_conci}")

    return valores


###############################################################
# LECTURA DE ESPECIES DESDE DENIT-10.xlsx
###############################################################

def leer_especies_desde_excel_denit(paso):
    """
    Lee datos de especies desde DENIT-10.xlsx.

    Rangos usados:
        paso = 1   -> C24:L33
        paso = 5   -> C49:L58
        paso = 10  -> C74:L83
        paso = 100 -> C99:L108

    Cada renglón del rango corresponde a una especie.
    """

    archivo_excel = "DENIT-10.xlsx"

    if not os.path.exists(archivo_excel):
        print(f"⚠️ No existe {archivo_excel} en el directorio actual.")
        return None

    rangos = {
        1:   ("C24:L33", 23, 32),
        5:   ("C49:L58", 48, 57),
        10:  ("C74:L83", 73, 82),
        100: ("C99:L108", 98, 107)
    }

    if paso not in rangos:
        return None

    rango_excel, fila_ini, fila_fin = rangos[paso]

    try:
        df = pd.read_excel(
            archivo_excel,
            header=None
        )

        datos_excel = df.iloc[
            fila_ini:fila_fin+1,
            2:12
        ].to_numpy(dtype=float)

    except Exception as e:
        print(f"⚠️ No se pudieron leer los datos de {archivo_excel}: {e}")
        return None

    return datos_excel, rango_excel

###############################################################
# GRAFICAR u_new.out
###############################################################

import matplotlib.pyplot as plt
    
def graficar_u_new(ntiempo):
    paso = ntiempo + 1
    """
    Grafica:

    1. Todas las componentes
    2. Todas las especies

    almacenadas en u_new.out.
    """

    if not os.path.exists(archivo_u_new):
        print(f"No existe {archivo_u_new}")
        return

    with open(archivo_u_new, "r") as f:
        lineas = f.readlines()

        ###########################################################
    # Nombres de especies desde u_init.dat
    ###########################################################

    with open(archivo_u_init, "r") as f:
        lineas_uinit = f.readlines()

    nombres_especies = []

    for k in range(13, 21):      # renglones 13..20
        if len(lineas_uinit) >= k:
            nombres_especies.append(
                lineas_uinit[k-1].strip()
            )
            
    ###########################################################
    # COMPONENTES
    ###########################################################

    plt.figure(figsize=(10,6))

    for m in range(n_comp):

        nr = m + 2      # índice python del renglón 3+m

        datos = np.array(
            [float(x) for x in lineas[nr].split()]
        )

        # x = np.arange(len(datos)) ################# Componentes
        x_plot = np.linspace(
            0,
            row_length,
            len(datos)
        )

        plt.plot(
            x_plot,
            datos,
            marker="o",
            label=f"Comp {m+1}"
        )

    plt.xlabel("Distance (m)")
    plt.ylabel("Concentration")
    plt.title(
        f"COMPONENTES - Paso de tiempo {paso}"
    )

    plt.grid(True)
    plt.legend()
    plt.show()

    ###########################################################
    # ESPECIES
    ###########################################################
    
    plt.figure(figsize=(10,6))
    
    for m in range(n_comp + 1):
    
        # renglón físico = n_comp + 6 + m
        # índice Python   = n_comp + 5 + m
        nr = n_comp + 5 + m
    
        datos = np.array(
            [float(x) for x in lineas[nr].split()]
        )
    
        # x = np.arange(len(datos)) ###############
        x_plot = np.linspace(
            0,
            row_length,
            len(datos)
        )
        
        ########################
        nombre = (
            nombres_especies[m]
            if m < len(nombres_especies)
            else f"Especie {m+1}"
        )
        
        plt.plot(
            x_plot,
            datos,
            marker="o",
            label=nombre
        )
    
    plt.xlabel("Distance (m)")
    plt.ylabel("Concentration")
    plt.title(
        f"ESPECIES - Paso de tiempo {paso}"
    )
    
    plt.grid(True)
    plt.legend()
    plt.show()

    ###########################################################
    # ESPECIES DESDE DENIT-10.xlsx
    ###########################################################

    resultado_excel = leer_especies_desde_excel_denit(paso)

    if resultado_excel is not None:

        datos_excel, rango_excel = resultado_excel

        plt.figure(figsize=(10,6))

        n_especies_excel = min(
            n_comp + 1,
            datos_excel.shape[0]
        )

        for m in range(n_especies_excel):

            datos_ref = datos_excel[m, :]

            mascara = np.isfinite(datos_ref)

            if not np.any(mascara):
                continue

            x_excel = np.linspace(
                0,
                row_length,
                len(datos_ref)
            )

            nombre = (
                nombres_especies[m]
                if m < len(nombres_especies)
                else f"Especie {m+1}"
            )

            plt.plot(
                x_excel[mascara],
                datos_ref[mascara],
                marker="x",
                linestyle="--",
                linewidth=2,
                label=nombre
            )

        plt.xlabel("Distance (m)")
        plt.ylabel("Concentration")
        plt.title(
            f"ESPECIES DESDE DENIT-10.xlsx ({rango_excel}) - Paso de tiempo {paso}"
        )

        plt.grid(True)
        plt.legend()
        plt.show()

        print(
            f"✅ Datos Excel graficados desde DENIT-10.xlsx, rango {rango_excel}"
        )
###############################################################
# BLOQUE 12. CICLO PRINCIPAL
###############################################################

def ciclo_denit():
    """
    Secuencia completa:

    PRIMERA PARTE:
    1. Lee u_init.dat.
    2. Para m = 0,...,n_comp-1:
       - lee líneas 24+m y 44+m
       - guarda ambas como columna
       - copia la primera columna a conc_i.m3d
       - define conc_inicial y contenido_44
       - ejecuta GWF-V5 y GWT-V5
       - obtiene cobj = o_gwt.output.concentration()
       - sustituye renglón m+1 de u_tilde.dat
    3. Ejecuta interfaz.

    SEGUNDA PARTE:
    4. Repite nv veces el proceso:
       - para m = 0,...,n_comp-1:
         * lee línea 3+m de u_new.out
         * la convierte en columna
         * la guarda en conc_i.m3d
         * ejecuta GWF-V5 y GWT-V5
         * obtiene cobj = o_gwt.output.concentration()
         * sustituye renglón m+1 de u_tilde.dat
       - ejecuta interfaz al terminar todos los componentes
    """

    output_area.clear_output()

    with output_area:

        ip = get_ipython()

        print("🟢 INICIANDO CICLO DENIT")
        print(f"n_comp = {n_comp}")
        print(f"nv     = {nv}")
        print(f"nplot  = {nplot}")
        print(f"ncolt  = {ncolt}")

        ###########################################################
        # PRIMERA EJECUCIÓN DESDE u_init.dat
        ###########################################################

        print("\n" + "#"*70) #############################
        print("# PRIMERA EJECUCIÓN DESDE u_init.dat")
        print("#"*70)        #############################

        i = 0

        for m in range(n_comp):
            
            # EJECUTAMOS: preparar_componente_desde_u_init
            preparar_componente_desde_u_init(m)

            # EJECUTAMOS: ejecutar_gwf_gwt 
            conc_flat = ejecutar_gwf_gwt(
                ip=ip,
                m=m,
                i=i
            )

            sustituir_renglon_u_tilde_desde_valores(
                valores=conc_flat,
                archivo_u_tilde=archivo_u_tilde,
                m=m
            )

        ###########################################################
        # PRIMERA EJECUCIÓN DE INTERFAZ
        ###########################################################

        ejecutar_interfaz()

        ###########################################################
        # Para graficar componentes y especies
        ###########################################################
        if i % nplot == 0:
            graficar_u_new(i)

        ###########################################################
        # REPETICIONES DESDE u_new.out
        ###########################################################

        #for i in range(1, nv + 1):
        for i in range(1, nv):

            print("\n" + "#"*70)
            print(f"# PASO DESDE u_new.out: i = {i} de {nv}")
            print("#"*70)

            for m in range(n_comp):

                ###################################################
                # Mantener contenido_44 del componente m
                ###################################################

                preparar_componente_desde_u_init(m)

                ###################################################
                # Leer línea 3+m de u_new.out y actualizar conc_i.m3d
                ###################################################

                preparar_componente_desde_u_new(m)

                ###################################################
                # Ejecutar GWF/GWT
                ###################################################

                conc_flat = ejecutar_gwf_gwt(
                    ip=ip,
                    m=m,
                    i=i
                )

                ###################################################
                # Sustituir renglón m+1 de u_tilde.dat
                ###################################################

                sustituir_renglon_u_tilde_desde_valores(
                    valores=conc_flat,
                    archivo_u_tilde=archivo_u_tilde,
                    m=m
                )

            ###########################################################
            # Ejecutar interfaz al terminar todos los componentes
            ###########################################################

            ejecutar_interfaz()

            paso = i + 1
            
            if paso % nplot == 0:
                graficar_u_new(i)
                                          
            print("\n✅ Fin del paso.")

            time.sleep(1)

        print("\n✅✅✅ CICLO DENIT TERMINADO")


###############################################################
# BLOQUE 13. BOTÓN DE EJECUCIÓN
###############################################################

run_button = widgets.Button(
    description="Iniciar ciclo",
    style=dict(button_color="lightgray")
)

output_area = widgets.Output()

run_button.on_click(lambda b: ciclo_denit())

display(run_button, output_area)


Button(description='Iniciar ciclo', style=ButtonStyle(button_color='lightgray'))

Output()